# Advisor Simulation Validation

Quick checks for the synthetic advisor, market, allocation, and monthly flow outputs.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('data')

advisors = pd.read_csv(DATA_DIR / 'advisors.csv')
market = pd.read_csv(DATA_DIR / 'market_returns.csv')
allocations = pd.read_csv(DATA_DIR / 'allocations.csv')
flows = pd.read_csv(DATA_DIR / 'monthly_flows.csv')

for name, df in {
    'advisors': advisors,
    'market': market,
    'allocations': allocations,
    'flows': flows,
}.items():
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

## Shape Checks

In [ ]:
expected_advisors = 1000
expected_months = 24
expected_categories = market['mstar_category'].nunique()

checks = {
    'advisor_rows': len(advisors) == expected_advisors,
    'market_rows': len(market) == expected_months * expected_categories,
    'allocation_rows': len(allocations) == expected_advisors * expected_months * expected_categories,
    'flow_rows': len(flows) == expected_advisors * expected_months * expected_categories,
    'flow_missing_assets': flows['assets'].isna().sum() == 0,
}

pd.Series(checks, name='passed')

## Advisor Mix

In [ ]:
advisor_mix = pd.concat(
    [
        advisors['archetype_id'].value_counts(normalize=True).rename('archetype_share'),
        advisors['channel'].value_counts(normalize=True).rename('channel_share'),
    ],
    axis=1,
)

advisor_mix.round(3)

In [ ]:
advisors['aum'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0)

## Market Regimes

In [ ]:
monthly_regimes = market[['month', 'market_regime', 'volatility_regime']].drop_duplicates()

display(monthly_regimes)
display(monthly_regimes['market_regime'].value_counts())
display(market.groupby(['market_regime', 'asset_class'])['return'].mean().unstack().round(4))

## Allocation Checks

In [ ]:
weight_sums = allocations.groupby(['advisor_id', 'month'])['allocation_pct'].sum()

pd.Series(
    {
        'min_weight_sum': weight_sums.min(),
        'max_weight_sum': weight_sums.max(),
        'mean_abs_error': (weight_sums - 1.0).abs().mean(),
    }
).round(10)

In [ ]:
allocation_joined = allocations.merge(advisors[['advisor_id', 'archetype_id']], on='advisor_id')
allocation_joined = allocation_joined.merge(
    market[['mstar_category', 'asset_class']].drop_duplicates(), on='mstar_category'
)
first_month = allocation_joined['month'].min()
allocation_asset_mix = (
    allocation_joined[allocation_joined['month'] == first_month]
    .groupby(['advisor_id', 'archetype_id', 'asset_class'])['allocation_pct']
    .sum()
    .groupby(['archetype_id', 'asset_class'])
    .mean()
    .unstack()
)

allocation_asset_mix.round(3)

## Flow Accounting

In [ ]:
net_sales_error = ((flows['gross_sales'] - flows['redemptions']) - flows['net_sales']).abs()

ordered_flows = flows.sort_values(['advisor_id', 'mstar_category', 'month']).reset_index(drop=True)
ordered_allocations = allocations.sort_values(['advisor_id', 'mstar_category', 'month']).reset_index(drop=True)
prior_assets = ordered_flows.groupby(['advisor_id', 'mstar_category'])['assets'].shift(1)
prior_assets = prior_assets.where(prior_assets.notna(), ordered_allocations['category_assets'])
expected_assets = prior_assets * (1 + ordered_flows['category_return']) + ordered_flows['net_sales']
asset_error = (expected_assets - ordered_flows['assets']).abs()

pd.Series(
    {
        'max_net_sales_error': net_sales_error.max(),
        'max_asset_equation_error': asset_error.max(),
        'mean_asset_equation_error': asset_error.mean(),
        'negative_assets': (flows['assets'] < 0).sum(),
    }
).round(6)

In [ ]:
flows[['gross_sales', 'redemptions', 'net_sales', 'assets']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(2)

In [ ]:
flows.groupby('market_regime')[['gross_sales', 'redemptions', 'net_sales']].mean().round(2)

In [ ]:
# flows.groupby(['advisor_id', 'month']).agg({'net_sales': 'sum', 'assets': 'sum'}).reset_index().to_csv('test.csv')